### ЗАДАЧА: Реестр пропусков коворкинга

Администратор коворкинга ведёт реестр пропусков резидентов.
Нужно собрать модель, которая позволит:
- загрузить пропуска в единый реестр,
- посмотреть только активные пропуска,
- отфильтровать резидентов по тарифу,
- посчитать суммарный остаток дней,
- понять, как меняется реестр после паузы, списания дня и продления.

В данных есть разные тарифы и статусы,
поэтому важно корректно валидировать plan, status и изменение состояния объекта.
        


In [5]:
# rows: pass_id|member_name|plan|days_left|status
rows = [
    'PS-100|Alice|flex|12|active',
    'PS-101|Bob|fixed|20|paused',
    'PS-102|Team Rocket|team|0|expired',
    'PS-103|Diana|flex|6|active',
]


class CoworkingPass:
    allowed_plans = {'flex', 'fixed', 'team'}
    allowed_statuses = {'active', 'paused', 'expired'}

    def __init__(self, pass_id, member_name, plan, days_left, status):
        # TODO: проверить plan и status, иначе raise ValueError(...)
        # TODO: сохранить pass_id, member_name, plan, status
        # TODO: days_left хранить через внутреннее поле self._days_left
        # TODO: значение days_left пропустить через property/setter
        if plan not in self.allowed_plans:
            raise ValueError('некорректный plan!')
        if status not in self.allowed_statuses:
            raise ValueError('неккоректный status!')
        self.pass_id = pass_id
        self.member_name = member_name
        self.plan = plan
        self.status = status
        self._days_left = days_left



    @property
    def days_left(self):
        # TODO: вернуть текущее число оставшихся дней
        return self._days_left

    @days_left.setter
    def days_left(self, value):
        # TODO: привести value к int
        # TODO: если value < 0 -> raise ValueError('Days must be >= 0')
        # TODO: сохранить результат в self._days_left
        value = int(value)
        if value < 0:
            raise ValueError('Количество дней должно быть больше 0!')
        self._days_left = value

    def use_day(self):
        # TODO: если статус не 'active' -> raise ValueError(...)
        # TODO: если days_left == 0 -> raise ValueError(...)
        # TODO: уменьшить days_left на 1
        # TODO: если после списания days_left == 0, перевести статус в 'expired'
        if self.status != 'active':
            raise ValueError('status должен быть active!')
        if self._days_left == 0:
            raise ValueError('Количество дней уже равно 0!')
        self._days_left -= 1
        if self._days_left == 0:
            self.status = 'expired'

    def pause(self):
        # TODO: если статус 'expired' -> raise ValueError(...)
        # TODO: перевести пропуск в 'paused'
        if self.status == 'expired':
            raise ValueError('статус expired!')
        self.status = 'paused'

    def resume(self):
        # TODO: если days_left == 0 -> raise ValueError(...)
        # TODO: перевести пропуск в 'active'
        if self._days_left == 0:
            raise ValueError('Количество дней равно 0!')
        self.status = 'active'

    def renew(self, extra_days):
        # TODO: привести extra_days к int
        # TODO: если extra_days <= 0 -> raise ValueError(...)
        # TODO: увеличить days_left
        # TODO: если days_left > 0 и статус был 'expired', перевести в 'active'
        extra_days = int(extra_days)
        if extra_days <= 0:
            raise ValueError('extra_days должно быть больше 0!')
        self._days_left += extra_days
        if self._days_left > 0 and self.status == 'expired':
            self.status = 'active'


    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        # TODO: ожидать 5 частей: pass_id, member_name, plan, days_left, status
        # TODO: вернуть CoworkingPass(...)
        arr = row.split('|')
        pass_id, member_name, plan, days_left, status = arr[0], arr[1], arr[2], int(arr[3]), arr[4]
        return CoworkingPass(pass_id=pass_id, member_name=member_name, status=status, days_left=days_left, plan=plan)
    def __repr__(self):
        # TODO: вернуть строку вида CoworkingPass(pass_id='...', member_name='...', status='...', days_left=...)
        return f'CoworkingPass(pass_id={self.pass_id}, member_name={self.member_name}, status={self.status}, days_left={self._days_left})'


class CoworkingRegistry:
    def __init__(self):
        self.items = []

    def add(self, coworking_pass):
        # TODO: добавить объект в self.items
        self.items.append(coworking_pass)

    def load(self, rows):
        # TODO: для каждой строки создать CoworkingPass.from_row(row)
        # TODO: добавить объект в реестр через add(...)
        for el in rows:
            self.add(CoworkingPass.from_row(el))

    def active_passes(self):
        # TODO: вернуть список пропусков со статусом 'active'
        res = []
        for el in self.items:
            if el.status == 'active':
                res.append(el)
        return res

    def by_plan(self, plan):
        # TODO: вернуть список пропусков нужного тарифа
        return [el for el in self.items if el.plan ==plan]

    def total_days_left(self):
        # TODO: вернуть суммарное число оставшихся дней
        c = 0
        for el in self.items:
            c += el._days_left
        return c

    def status_summary(self):
        # TODO: собрать dict вида status -> count
        obj = {}
        for el in self.items:
            obj.setdefault(el.status, 0)
            obj[el.status] += el._days_left
        return obj
    def find(self, pass_id):
        # TODO: вернуть пропуск по pass_id или None
        for el in self.items:
            if el.pass_id == pass_id:
                return el
        return None

    def largest_balance(self):
        # TODO: найти пропуск с максимальным days_left
        # TODO: вернуть tuple(pass_id, days_left)
        largest_id = None
        largest_days = 0
        for el in self.items:
            if el._days_left > largest_days:
                largest_days = el._days_left
                largest_id = el.pass_id
        return (largest_id, largest_days)


registry = CoworkingRegistry()

# TODO: загрузить rows в registry
# TODO: вывести все пропуска
# TODO: вывести active_passes()
# TODO: вывести by_plan('flex')
# TODO: вывести total_days_left()
# TODO: вывести status_summary()
# TODO: найти пропуск 'PS-101', возобновить его и вывести status_summary()
# TODO: найти пропуск 'PS-100', списать один день и вывести объект
# TODO: найти пропуск 'PS-102', продлить на 5 дней и вывести объект
# TODO: вывести largest_balance()
        
registry.load(rows)
print('Все пропуски: ', registry.items)
print('активные: ', registry.active_passes())
print('с планом flex: ', registry.by_plan('flex'))
print('сумма дней: ', registry.total_days_left())
print(registry.status_summary())
res = registry.find('PS-101')
res.resume()
print('После возобновления PS-101: ', registry.status_summary())
res = registry.find('PS-100')
res.use_day()
print(res)
res = registry.find('PS-102')
res.renew(5)
print(res)
print(registry.largest_balance())

Все пропуски:  [CoworkingPass(pass_id=PS-100, member_name=Alice, status=active, days_left=12), CoworkingPass(pass_id=PS-101, member_name=Bob, status=paused, days_left=20), CoworkingPass(pass_id=PS-102, member_name=Team Rocket, status=expired, days_left=0), CoworkingPass(pass_id=PS-103, member_name=Diana, status=active, days_left=6)]
активные:  [CoworkingPass(pass_id=PS-100, member_name=Alice, status=active, days_left=12), CoworkingPass(pass_id=PS-103, member_name=Diana, status=active, days_left=6)]
с планом flex:  [CoworkingPass(pass_id=PS-100, member_name=Alice, status=active, days_left=12), CoworkingPass(pass_id=PS-103, member_name=Diana, status=active, days_left=6)]
сумма дней:  38
{'active': 18, 'paused': 20, 'expired': 0}
После возобновления PS-101:  {'active': 38, 'expired': 0}
CoworkingPass(pass_id=PS-100, member_name=Alice, status=active, days_left=11)
CoworkingPass(pass_id=PS-102, member_name=Team Rocket, status=active, days_left=5)
('PS-101', 20)
